<a href="https://colab.research.google.com/github/RomanPhys/External/blob/main/sasrec_external_context.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# SASRec авторов и разложение внешнего контекста

Ноутбук делает четыре вещи:

1. Обучает SASRec из репозитория авторов на MovieLens-1M. Эпоха выбирается по валидационным пользователям, а не по тестовым.
2. Повторяет контроль «лишние эпохи без контекста» на настоящем SASRec.
3. Выгружает состояния SASRec и прогоняет на них весь конвейер разложения: агрегации статьи, «похожие клиенты» против «контекста в целом», retrieval переходов и контроль g = 0.
4. По желанию запускает агрегации авторов в их собственном коде с `freeze=False`, с тем же числом эпох, что у контроля.

Время на бесплатной T4: шаги 1–3 занимают около 40 минут, шаг 4 ещё примерно 20 минут на каждую агрегацию.

**Перед запуском:** Runtime → Change runtime type → T4 GPU.

In [1]:
!nvidia-smi -L
import torch; print("torch", torch.__version__, "cuda", torch.cuda.is_available())

GPU 0: Tesla T4 (UUID: GPU-c9c71722-fc75-3b3a-6ce5-988df753319f)
torch 2.14.1+cu130 cuda True


## 1. Код из архива

Загрузи `external_context_v3.zip`. Если сессия может оборваться, поставь `USE_DRIVE = True`: результаты будут сохраняться на Google Drive, и шаги можно будет продолжить с места остановки.

In [2]:
import os, zipfile
USE_DRIVE = False
if USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    WORK = "/content/drive/MyDrive/ext_work"
else:
    WORK = "/content/work"
os.makedirs(WORK, exist_ok=True)

if not os.path.exists("/content/ext/code/run_context.py"):
    from google.colab import files
    up = files.upload()                      # выбери external_context_v3.zip
    zipfile.ZipFile(next(iter(up))).extractall("/content/ext")
print(sorted(os.listdir("/content/ext")), "WORK =", WORK)

Saving external_context_v3.zip to external_context_v3.zip
['README.md', 'code', 'colab', 'results', 'torch_patch'] WORK = /content/work


## 2. Репозиторий авторов и данные

Пакет `recsys/src/data` выпал из репозитория авторов из-за правила `data/` в `.gitignore`. Здесь он восстанавливается из `torch_patch/recsys_src_data.py`. Данные берутся с сайта GroupLens. Должно получиться 1000209 строк.

In [3]:
%cd /content
!test -d eca || git clone -q --depth 1 https://github.com/petrsokerin/External-Context-Aggregation.git eca
!mkdir -p eca/recsys/src/data eca/recsys/data/movielens_1m
!cp /content/ext/torch_patch/recsys_src_data.py eca/recsys/src/data/__init__.py
!cp /content/ext/torch_patch/sasrec_pipeline.py eca/recsys/
!test -f /content/ml-1m.zip || wget -q https://files.grouplens.org/datasets/movielens/ml-1m.zip -O /content/ml-1m.zip
!unzip -o -q /content/ml-1m.zip -d /content/
!cp /content/ml-1m/ratings.dat eca/recsys/data/movielens_1m/
!wc -l eca/recsys/data/movielens_1m/ratings.dat

/content
1000209 eca/recsys/data/movielens_1m/ratings.dat


## 3. Предобучение SASRec, выгрузка состояний, контроль лишних эпох

Гиперпараметры берутся из `config/main_movielens_1m.yaml` авторов: 150 эпох, AdamW, lr 0.001, max_len 200.

Тестовые пользователи авторов (20%) делятся на val (1/3) и test (2/3). Лучшая эпоха выбирается по val. После этого модель ещё 20 эпох учится без контекста с Adam, как второй этап авторов. Это контроль для шага 6.

Что должно получиться: строка `pretrained` с val и test в районе 0.07–0.09 (у авторов на их сплите 0.0776), затем 20 строк `no-context` и файл `sasrec_states.npz`. Около 15–25 минут.

In [4]:
%cd /content/eca/recsys
!python sasrec_pipeline.py --config config/main_movielens_1m.yaml --work {WORK} --extra_epochs 20

/content/eca/recsys
interactions 574376, items 3125, users {'train': 4827, 'val': 402, 'test': 805}
True
Epoch 1/150: 100% 151/151 [00:03<00:00, 43.71it/s]
Epoch 2/150: 100% 151/151 [00:02<00:00, 53.34it/s]
Epoch 3/150: 100% 151/151 [00:02<00:00, 64.89it/s]
2
100% 13/13 [00:00<00:00, 51.33it/s]
Epoch 3, Tr Loss: 7.8659, Val Loss 7.6155, Val NDCG@10: 0.0062
Epoch 4/150: 100% 151/151 [00:02<00:00, 64.79it/s]
Epoch 5/150: 100% 151/151 [00:02<00:00, 64.97it/s]
Epoch 6/150: 100% 151/151 [00:02<00:00, 64.20it/s]
5
100% 13/13 [00:00<00:00, 37.40it/s]
Epoch 6, Tr Loss: 7.3502, Val Loss 7.4787, Val NDCG@10: 0.0116
Epoch 7/150: 100% 151/151 [00:02<00:00, 53.03it/s]
Epoch 8/150: 100% 151/151 [00:02<00:00, 64.62it/s]
Epoch 9/150: 100% 151/151 [00:02<00:00, 64.49it/s]
8
100% 13/13 [00:00<00:00, 53.18it/s]
Epoch 9, Tr Loss: 7.2671, Val Loss 7.4679, Val NDCG@10: 0.0126
Epoch 10/150: 100% 151/151 [00:02<00:00, 64.08it/s]
Epoch 11/150: 100% 151/151 [00:02<00:00, 64.33it/s]
Epoch 12/150: 100% 151/151 [0

## 4. Разложение на состояниях SASRec

Тот же numpy-конвейер, что в отчёте, но вместо MS-EMA он читает выгрузку SASRec. Идёт на CPU, около 8 минут.

- `run_experiments.py` повторяет агрегации статьи, раскладывает эффект на «похожих» и «контекст в целом» и проверяет шлюз.
- `run_shrink_control.py` делает контроль g = 0.
- `run_diagnostics.py` считает производные CE по доле контекста.

In [5]:
os.environ["EXT_STATES"] = f"{WORK}/sasrec_states.npz"
os.environ["EXT_WORK"] = WORK
%cd /content/ext/code
!python run_context.py
!python run_experiments.py > {WORK}/experiments.stdout 2>&1; tail -n 5 {WORK}/experiments.stdout
!python run_shrink_control.py
!python run_diagnostics.py > {WORK}/diagnostics.log 2>&1; echo diagnostics done

/content/ext/code
state rows 460248 queries {'train': 28923, 'val': 2008, 'test': 4023} 1s
sweep done 47s
full retrieval done 79s
saved 96s
val active pool size at query time: median 2872, 10% 507
test active pool size at query time: median 2975, 10% 719
authors' fixed pool (100): active median 58, 10% 8
[C_new] NEW Шлюз + retrieval (lam=0.2)         last 0.1080 all 0.1166  d_all +0.0057 CI [+0.0020,+0.0097] p=0.004  lam=0.2
by length Шлюз local/global [{'bin': 'короткая', 'n_users': 287, 'base': 0.1548913519708567, 'gain': 0.001014319560276399, 'ci': (np.float64(-0.004430173236685672), np.float64(0.006180046168369745))}, {'bin': 'средняя', 'n_users': 284, 'base': 0.10651937756753986, 'gain': 0.0023798922090340923, 'ci': (np.float64(-0.0024896118471115936), np.float64(0.008026944287415308))}, {'bin': 'длинная', 'n_users': 272, 'base': 0.06944455001437767, 'gain': 0.0031495125872521255, 'ci': (np.float64(-0.0003612527372805618), np.float64(0.006828871683292121))}]
by length Retrieval вс

## 5. Память переходов

Здесь причинный retrieval и все его варианты: похожие клиенты, похожие ситуации, окна по времени, утечка, случайный контроль, векторная форма и адаптивная λ. Около 10–12 минут на CPU.

In [6]:
%cd /content/ext/code
!python run_retrieval.py > {WORK}/retrieval.stdout 2>&1; tail -n 3 {WORK}/retrieval.stdout
!python summarize.py

/content/ext/code
adaptive {'ndcg_all': 0.1140237534064874, 'd_all': 0.0032131105324568637, 'ci_all': (np.float64(0.0012110163913327907), np.float64(0.005208024797779406)), 'p_all': np.float64(0.002), 'ndcg_last': 0.1049980234852067, 'd_last': 8.614379280558249e-05, 'ci_last': (np.float64(-0.0037225726151017863), np.float64(0.004053201561109746)), 'vs_fixed': 0.0001912047102032331, 'vs_fixed_ci': (np.float64(-0.00302434123154888), np.float64(0.0033193200533574493)), 'vs_fixed_p': np.float64(0.903), 'weights': {'top1 cos': 0.6946984848494357, 'mean top10 cos': -0.8055974074739064, 'энтропия весов': 1.0863796549679556, 'max p_knn': 0.23449363921541744, 'max p_model': -0.055278510445727104, 'энтропия p_model': -0.2689309294125908, 'log длины': 0.016331596752180225, 'log паузы (ч)': -0.6013169291438433}, 'lam_mean': 0.07895479799767852, 'lam_q10': 0.04220816425175679, 'lam_q90': 0.12061305093224814, 'K': 100, 'tau': 5.0, 'lam_fixed': 0.3, 'train_nll': 5.288431271096359, 'train_nll_fixed': 

## 6. По желанию: агрегации авторов в их коде

Это `add_external_features` авторов с `freeze=False` и α = 0.1, 20 эпох, эпоха выбирается по val. Модель с контекстом сравнивается и с предобученной моделью, и с контролем «те же 20 эпох без контекста» из шага 3. Позиции раньше первого события пула получают пустой контекст: у авторов индекс −1 в `agg.py` указывает на будущее. Скрипт печатает, какая доля обучающих позиций и тестовых запросов попала бы под это в исходном коде.

Около 20 минут на каждую агрегацию. Если времени мало, оставь только `learnable_attention`.

In [7]:
%cd /content/eca/recsys
!python sasrec_pipeline.py --config config/main_movielens_1m.yaml --work {WORK} --skip_pretrain --extra_epochs 0 --authors_agg learnable_attention,mean --freeze 0 --alpha 0.1 --ext_epochs 20
%cd /content/ext/code
!python summarize.py

/content/eca/recsys
interactions 574376, items 3125, users {'train': 4827, 'val': 402, 'test': 805}
100% 26/26 [00:00<00:00, 43.81it/s]
pretrained: {'val': 0.12133952230215073, 'test': 0.10996947437524796, 'test_authors_metric': 0.10964594550252903}
Processing users: 100% 100/100 [00:00<00:00, 220.21it/s]
Aggregating embeddings: 100% 4805/4805 [00:03<00:00, 1378.31it/s]
  [learnable_attention] epoch 1: loss 5.5906 val 0.1111 test 0.1092 (24s)
  [learnable_attention] epoch 2: loss 5.5868 val 0.1141 test 0.1047 (19s)
  [learnable_attention] epoch 3: loss 5.5800 val 0.1156 test 0.1085 (20s)
  [learnable_attention] epoch 4: loss 5.5819 val 0.1120 test 0.1036 (20s)
  [learnable_attention] epoch 5: loss 5.5756 val 0.1092 test 0.1080 (20s)
  [learnable_attention] epoch 6: loss 5.5739 val 0.1147 test 0.1089 (20s)
  [learnable_attention] epoch 7: loss 5.5716 val 0.1188 test 0.1070 (20s)
  [learnable_attention] epoch 8: loss 5.5691 val 0.1104 test 0.1099 (20s)
  [learnable_attention] epoch 9: lo

## 7. Сохранить и прислать

Архив с JSON и логами без больших файлов. Пришли его мне вместе с выводом `summarize.py`.

In [8]:
!cd {WORK} && zip -q -r /content/results_sasrec.zip . -x "*.npz" "*.pkl" "*.pth"
from google.colab import files
files.download("/content/results_sasrec.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## Как читать результаты

- **Retrieval переходов значим, Mean и Learnable attention около нуля.** Тогда вывод с ML-1M переносится на трансформер. Если прирост retrieval меньше, чем на MS-EMA, это ожидаемо: сильной модели остаётся меньше, что исправлять.
- **Агрегация авторов выше предобученного SASRec, но не выше контроля лишних эпох.** Тогда прирост в статье объясняется протоколом, а не контекстом.
- **Контроль g = 0 даёт столько же, сколько G.** Тогда «контекст в целом» это калибровка, как на ритейле.
- **Производная по G положительна, по retrieval отрицательна.** Тогда диагностика до обучения работает и на SASRec.

**Если что-то пошло не так.**

- `ModuleNotFoundError: src.data`: не выполнилась ячейка шага 2, запусти её снова.
- CUDA out of memory на шаге 6: добавь `--eval_batch 16`.
- Сессия оборвалась: включи `USE_DRIVE = True` с самого начала. Тогда после переподключения выполни шаги 1 и 2 и продолжай с того шага, на котором остановился. Чекпойнт, выгрузка и контроль лишних эпох лежат в `WORK` на Drive, шаг 6 их подхватит через `--skip_pretrain`.
- Шаги 4 и 5 можно перезапускать по отдельности. Им нужен только `sasrec_states.npz` и кеш `cache_context.pkl` в `WORK`.